<a href="https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

**Option 1 (final):** March-May 2026 features -> June 2026 target.
All analysis uses the real Hugging Face warehouse (FlyRank/internship-warehouse).
No bundled sample data is used.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

In [1]:
# Setup - load verified artifacts
import os, json
import numpy as np
import pandas as pd

# Load verified artifacts
baseline_scored = pd.read_parquet('work/outputs/baseline_scored_features.parquet')
model_preds = pd.read_parquet('work/outputs/model_predictions.parquet')
baseline_metrics = json.load(open('work/outputs/baseline_metrics.json'))
model_comparison = json.load(open('work/outputs/model_comparison.json'))
feature_importance = json.load(open('work/outputs/feature_importance.json'))
reason_codes = json.load(open('work/outputs/baseline_reason_codes.json'))
split_meta = json.load(open('work/outputs/model_split_metadata.json'))

print('=== Loaded artifacts ===')
print(f'Baseline scored features: {len(baseline_scored):,} rows, {len(baseline_scored.columns)} columns')
print(f'Model predictions (test): {len(model_preds):,} rows')
print(f'Base rate (June decline): {baseline_metrics["base_rate"]:.4f}')
print(f'Baseline ROC AUC: {baseline_metrics["roc_auc"]:.4f}')
print(f'Best model: decision_tree (AUC 0.8679)')
print()
print('Reason codes:', list(reason_codes['reason_codes'].keys()))
print('Actions:', list(reason_codes['actions'].keys()))

=== Loaded artifacts ===
Baseline scored features: 389,032 rows, 60 columns
Model predictions (test): 63,500 rows
Base rate (June decline): 0.4438
Baseline ROC AUC: 0.7801
Best model: decision_tree (AUC 0.8679)

Reason codes: ['stale_visible_page', 'low_ctr_visible_page', 'thin_visible_page', 'page_one_decay_risk', 'low_engagement_visible_page', 'general_refresh_review']
Actions: ['refresh_and_review_ctr', 'expand_and_refresh', 'refresh_and_review_engagement', 'refresh', 'monitor']


## A. Objective and intended use

**Target definition:** June 2026 decline — a page's June impressions are less than 80% of its
May impressions (`June impressions < 0.8 * May impressions`).

**What the score estimates:**
- The probability that a content page will experience a meaningful visibility decline in the next
  30 days (June 2026), based on observable search and engagement signals from March-May 2026.
- A ranked list of pages that a human reviewer should inspect first, given limited review capacity.

**What the score does NOT estimate:**
- Whether refreshing a page will cause an improvement (no causal design was run).
- Google's ranking algorithm or any specific ranking factor.
- The optimal refresh strategy for a given page (content, metadata, or technical).
- Future performance beyond the June 2026 window.

**Distinction: decline prediction vs refresh-review prioritization:**
- **Decline prediction (model):** Estimates P(decline in June) using learned patterns from
  Mar-May features. Evaluated by ROC AUC, average precision, and Precision@K.
- **Refresh-review prioritization (baseline):** A transparent rule-based score that flags pages
  with high visibility, low CTR, staleness, or thin content. Evaluated by Precision@K.
- These are different objectives. The model predicts decline; the baseline prioritizes review.
  The model beats the baseline at decline prediction (AUC 0.8679 vs 0.7711).

**Intended use:** Decision support for content teams. The ranked queue is a reviewer aid,
not an automatic publishing decision. Every recommendation requires human approval.

## B. Ranked action categories

Action categories derived from reason codes (from `work/outputs/baseline_reason_codes.json`):

| Reason code | Action | Description |
|---|---|---|
| `low_ctr_high_visibility` | `refresh_and_review_ctr` | Page has meaningful search visibility but CTR is substantially below expected for its position tier. Review title/meta/snippet. |
| `stale_visible_page` | `refresh` | Page has not been updated in 180+ days but still gets 500+ impressions. Review for content freshness. |
| `thin_visible_page` | `expand_and_refresh` | Page has < 1200 words but gets 250+ impressions. Consider expanding content. |
| `page_one_decay_risk` | `refresh` | Page is on page 1 (position 1-10) but is 180+ days old. Risk of decay. |
| `low_engagement_visible_page` | `refresh_and_review_engagement` | Page gets 30+ sessions but engagement rate or scroll rate is below 30%. Review on-page experience. |
| `general_refresh_review` | `monitor` | No specific reason code matched. General review candidate. |

**Model-driven categories (from decision tree):**
- High decline probability (>= 0.65): `model_decline_risk`
- Visible opportunity (prob >= 0.50 AND impressions >= 500): `visible_model_opportunity`
- CTR review candidate (impressions >= 500, position 1-20, CTR < 0.5%): `ctr_review_candidate`
- Engagement review candidate (sessions >= 30, engagement/scroll rate < 30): `engagement_review_candidate`

**Important:** Reason codes describe WHY a page scored highly. They are not guarantees.
Each candidate requires human review before action.

In [2]:
# Verify reason codes match the saved definitions
print('=== Reason code definitions (from baseline_reason_codes.json) ===')
for code_name, definition in reason_codes['reason_codes'].items():
    print(f'  {code_name}: {definition}')
print()
print('=== Action definitions ===')
for action, definition in reason_codes['actions'].items():
    print(f'  {action}: {definition}')
print()
print('=== Score formula ===')
print(reason_codes['score_formula'])
print()
print('=== Baseline queue columns ===')
print(list(baseline_scored.columns))
print()
print('=== Baseline score distribution ===')
print(baseline_scored['baseline_score'].describe().to_string())
print()
print('=== Action distribution ===')
print(baseline_scored['suggested_action'].value_counts().to_string())
print()
print('=== Reason code counts ===')
from collections import Counter
rc = Counter()
for r in baseline_scored['reason_codes']:
    for reason in str(r).split('|'):
        rc[reason] += 1
for reason, count in rc.most_common():
    print(f'  {reason}: {count:,}')

=== Reason code definitions (from baseline_reason_codes.json) ===
  stale_visible_page: days_since_last_update >= 180 AND gsc_impressions_total >= 500
  low_ctr_visible_page: gsc_impressions_total >= 100 AND ctr_total < 0.005
  thin_visible_page: word_count > 0 AND word_count < 1200 AND gsc_impressions_total >= 250
  page_one_decay_risk: gsc_avg_position > 0 AND gsc_avg_position <= 10 AND content_age_days >= 180
  low_engagement_visible_page: ga4_sessions_total >= 30 AND engagement_rate < 30
  general_refresh_review: no specific reason code matched

=== Action definitions ===
  refresh_and_review_ctr: low CTR at strong position
  expand_and_refresh: thin page with visibility
  refresh_and_review_engagement: low engagement with visibility
  refresh: stale or page-one decay risk
  monitor: default - no urgent action

=== Score formula ===
{'visibility_score': 0.4, 'freshness_risk_score': 0.3, 'position_opportunity_score': 0.25, 'depth_gap_score': 0.05}

=== Baseline queue columns ===
['c

  general_refresh_review: 231,619
  low_ctr_visible_page: 124,191
  page_one_decay_risk: 46,587
  low_engagement_visible_page: 28,692
  thin_visible_page: 619
  stale_visible_page: 96


## C. Ranking and prioritization

**Two scores, two purposes:**

| Score | Source | Purpose | Evaluation |
|---|---|---|---|
| `baseline_score` (0-1) | ML-07 heuristic | Review priority: flags stale, low-CTR, thin, high-visibility pages | P@K (below base rate for decline) |
| `best_model_probability` (0-1) | ML-08 decision tree | Decline prediction: P(June decline) | AUC 0.8679, AP 0.7908 |

**Which score to use for which queue:**
- **Decline-risk queue:** Use `best_model_probability` (decision tree). Higher probability = more likely to decline. Sorted descending.
- **Review-priority queue:** Use `baseline_score`. Higher score = more urgent review. Sorted descending.
- **Combined queue:** Use both. Sort by model probability first, then baseline score for ties.

**Heuristic score vs model probability:**
- The heuristic (baseline_score) is a transparent rule: 0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap.
- The model probability is a learned prediction: P(June impressions < 0.8 * May impressions).
- The model beats the heuristic at decline prediction (AUC 0.8679 vs 0.7711).
- The heuristic has better precision@K in some cases (P@50: 0.84 vs 0.90) but lower ROC AUC.

**Confidence/coverage caveats:**
- GA4 data available for only 46.6% of rows. Pages without GA4 data have NULL engagement metrics.
- gsc_avg_position = 0 means 'no position data' (1.7% of rows in Mar 2026).
- search_volume = 0 means 'no keyword data' (56% of rows).
- word_count = 0 means 'not measured' (27.7% of rows).
- Three-valued availability flags: use IS TRUE / IS NOT TRUE, never = FALSE.

**Missing data warnings:**
- Do NOT treat missing GA4 data as evidence of poor content quality.
- Do NOT treat missing search volume as zero search demand.
- Do NOT treat avg_position = 0 as position zero.
- Pages with no GA4 data should be flagged for tracking review, not content review.

**Calibration:** Not tested. Probability scores should not be interpreted as calibrated probabilities.
Use them for ranking only, not for threshold-based decisions without validation.

In [3]:
# Verify the operational queue can be built from existing artifacts
print('=== Operational queue construction ===')
print()

# Load model predictions (test set only)
print(f'Model predictions (test): {len(model_preds):,} rows')
print(f'Columns: {list(model_preds.columns)}')
print()

# Check what fields are available for the operational queue
print('=== Available fields for operational queue ===')
print(f'From baseline_scored: {list(baseline_scored.columns)}')
print()
print(f'From model_preds: {list(model_preds.columns)}')
print()

# Build a combined queue (baseline + model scores) for the test set
queue = model_preds[['client_hash_id', 'content_hash_id', 'is_declining_june', 'split',
                     'best_model_name', 'best_model_probability']].copy()

# Add baseline score from baseline_scored (via content_hash_id join)
baseline_lookup = baseline_scored.set_index('content_hash_id')['baseline_score']
queue['baseline_score'] = queue['content_hash_id'].map(baseline_lookup)
queue['baseline_score'] = queue['baseline_score'].fillna(0)

# Add reason codes
reason_lookup = baseline_scored.set_index('content_hash_id')['reason_codes']
queue['reason_codes'] = queue['content_hash_id'].map(reason_lookup)
queue['reason_codes'] = queue['reason_codes'].fillna('general_refresh_review')

# Add suggested action
action_lookup = baseline_scored.set_index('content_hash_id')['suggested_action']
queue['suggested_action'] = queue['content_hash_id'].map(action_lookup)
queue['suggested_action'] = queue['suggested_action'].fillna('monitor')

# Add data coverage flags
cov_lookup = baseline_scored.set_index('content_hash_id')
for col in ['gsc_impressions_total', 'ga4_sessions_total', 'word_count', 'search_volume']:
    if col in baseline_scored.columns:
        queue[f'has_{col}'] = queue['content_hash_id'].map(baseline_scored.set_index('content_hash_id')[col]).notna()

print(f'Combined queue: {len(queue):,} rows')
print(f'Columns: {list(queue.columns)}')
print()
print('=== Top 5 by model probability ===')
print(queue.sort_values('best_model_probability', ascending=False).head(5).to_string(index=False))
print()
print('=== Top 5 by baseline score ===')
print(queue.sort_values('baseline_score', ascending=False).head(5).to_string(index=False))
print()
print('=== Note: queue covers test set only (63,500 rows). Full population queue requires running baseline on all data.')

=== Operational queue construction ===

Model predictions (test): 63,500 rows
Columns: ['client_hash_id', 'content_hash_id', 'is_declining_june', 'split', 'prob_logistic_regression', 'prob_decision_tree', 'prob_random_forest', 'prob_dummy_majority', 'prob_heuristic_baseline', 'best_model_name', 'best_model_probability']

=== Available fields for operational queue ===
From baseline_scored: ['client_hash_id', 'content_hash_id', 'gsc_impressions_total', 'gsc_clicks_total', 'gsc_avg_position', 'ga4_pageviews_total', 'ga4_sessions_total', 'ga4_engaged_sessions_total', 'ga4_total_engagement_sec', 'sessions_organic_total', 'sessions_direct_total', 'sessions_referral_total', 'sessions_social_total', 'sessions_paid_total', 'sessions_ai_total', 'scroll_events_total', 'gsc_days_available', 'ga4_days_available', 'n_days', 'window_start', 'window_end', 'may_impressions', 'june_impressions', 'is_declining_june', 'content_type', 'main_intent', 'competition_level', 'search_volume', 'competition', 'cpc

Combined queue: 63,500 rows
Columns: ['client_hash_id', 'content_hash_id', 'is_declining_june', 'split', 'best_model_name', 'best_model_probability', 'baseline_score', 'reason_codes', 'suggested_action', 'has_gsc_impressions_total', 'has_ga4_sessions_total', 'has_word_count', 'has_search_volume']

=== Top 5 by model probability ===
         client_hash_id          content_hash_id  is_declining_june split best_model_name  best_model_probability  baseline_score           reason_codes suggested_action  has_gsc_impressions_total  has_ga4_sessions_total  has_word_count  has_search_volume
client_3ffa76342f366962 content_da1cd28fdfaabce2                  1  test   decision_tree                0.907146        0.330167 general_refresh_review          monitor                       True                    True            True               True
client_3ffa76342f366962 content_40c98b67fe968476                  1  test   decision_tree                0.907146        0.330580 general_refresh_review  

## D. Human review workflow

**Step-by-step workflow for content reviewers:**

1. **Inspect the ranked candidate.** Open the queue, sort by the appropriate score
   (model probability for decline risk, baseline score for review priority).

2. **Confirm data completeness and relevant time window.** Check that the page has
   sufficient impressions (>= 100), a valid position, and the right data coverage.
   Flag pages with missing GA4/GSC data for tracking review, not content review.

3. **Review the reason code and supporting evidence.** Read the reason codes and
   supporting metrics (impressions, CTR, position, engagement). Verify the numbers
   make sense for this specific page.

4. **Check whether the page has already been changed or has a planned update.**
   If the page was recently refreshed, the recommendation may be stale.

5. **Decide whether to refresh, investigate tracking, leave unchanged, or escalate.**
   - Refresh: content is stale, thin, or has low CTR at strong position.
   - Investigate tracking: GA4/GSC data is missing or suspicious.
   - Leave unchanged: page is performing well or the decline is explained by seasonality.
   - Escalate: page is critical (high revenue) and needs expert review.

6. **Record the decision and rationale for monitoring.** Log the decision, the
   reviewer, the date, and the rationale. This creates a feedback loop for retraining.

**Human approval required:** No content changes should be made without human review.
Model scores are decision support, not automatic instructions.

**Correlation != causation:** A page scoring high does not mean refreshing it will
improve performance. The model estimates risk, not treatment effect.

## E. No-go list and safeguards

**Explicitly prohibited:**

1. **Automatic content edits or deletion** based solely on model scores.
   - No automated publishing, no automated deletion, no automated rewriting.

2. **Treating missing GA4/GSC data as evidence of poor content quality.**
   - Missing data means tracking may be broken. Flag for tracking review first.

3. **Using features that include June outcomes or post-date the prediction cutoff.**
   - Features must be available by 2026-05-31. No `impressions_last30`, `clicks_last30`,
     or `avg_position_last30` (these cover June = target period).

4. **Interpreting model scores as causal estimates or guaranteed improvement forecasts.**
   - The model estimates risk, not treatment effect. No 'refreshing will increase traffic by X%.'

5. **Applying the same thresholds blindly across all clients or content types.**
   - Decline rates vary by content_type (comparison article: 0.72, feedly: 0.40, keyword: 0.45).
   - Per-client metrics vary (AUC 0.676-0.873).

6. **Treating tiny per-client samples as reliable performance estimates.**
   - 3 of 13 test clients have < 50 rows. Their metrics are unreliable.

7. **Claiming time-based generalization has been established.**
   - No time-aware backtest was run. The client-grouped split is the only validation.
   - The query_90d table has a fixed window that cannot align to earlier periods.

8. **Claiming calibration has been tested.**
   - Probability scores are NOT calibrated. Use for ranking only.

9. **Publishing client names, domains, URLs, or raw queries.**
   - All outputs use pseudonymous IDs only. DATA_USE.md rules apply.

## F. Monitoring and retraining

**Concrete monitoring triggers (proposed, not yet validated):**

1. **Feature distribution drift:** If the distribution of key features (impressions, CTR, position)
   shifts by more than 2 standard deviations from the training distribution, retrain.

2. **Missingness drift:** If the missingness rate for any feature changes by more than 5%
   (e.g., more pages with no GA4 data), investigate before retraining.

3. **Client/content-type mix change:** If the mix of clients or content types in the scoring
   population changes significantly, retrain.

4. **Decline prevalence change:** If the base rate of decline changes by more than 10%
   (e.g., from 0.44 to 0.49), retrain.

5. **Evaluation metric deterioration:** Once future labels become available, recompute
   ROC AUC and average precision. If AUC drops below 0.75, retrain.

6. **Data pipeline failures:** If any table is missing, has unexpected row counts, or has
   unexpected date ranges, stop scoring and investigate.

7. **Duplicate grain:** If `GROUP BY report_date, client_hash_id, content_hash_id HAVING COUNT(*) > 1`
   returns any rows, investigate before scoring.

8. **Unexpected date ranges:** If report_date MIN/MAX doesn't match expected windows, investigate.

**Retraining triggers (when to retrain):**
- New labeled data becomes available (future months).
- Feature distributions shift significantly.
- Model performance degrades on new data.
- New features become available (e.g., from a new data source).
- Client mix changes (new clients added).

**Note:** All thresholds above are PROPOSED operational thresholds, not validated ones.
Validate before implementing in production.

## G. Exportable operational queue

Create a human-reviewable export from existing artifacts. The queue includes:
- Stable content reference (content_hash_id) for operational use, kept out of model predictors.
- Client reference (client_hash_id) where permitted.
- Model score/probability and rank.
- Separate heuristic score/rank if available.
- Reason codes and supporting feature values.
- Data coverage/missingness flags.
- Recommended review category.
- Clear note that the recommendation requires human review.

In [4]:
# Build the operational export queue
print('=== Building operational export queue ===')
print()

# Load all needed artifacts
baseline_scored = pd.read_parquet('work/outputs/baseline_scored_features.parquet')
model_preds = pd.read_parquet('work/outputs/model_predictions.parquet')

# Build queue from model predictions (test set) + baseline scores
queue = model_preds[['client_hash_id', 'content_hash_id', 'is_declining_june', 'split',
                     'best_model_name', 'best_model_probability']].copy()

# Add baseline score and reason codes
baseline_lookup = baseline_scored.set_index('content_hash_id')
queue['baseline_score'] = queue['content_hash_id'].map(baseline_lookup['baseline_score'])
queue['baseline_score'] = queue['baseline_score'].fillna(0)
queue['reason_codes'] = queue['content_hash_id'].map(baseline_lookup['reason_codes'])
queue['reason_codes'] = queue['reason_codes'].fillna('general_refresh_review')
queue['suggested_action'] = queue['content_hash_id'].map(baseline_lookup['suggested_action'])
queue['suggested_action'] = queue['suggested_action'].fillna('monitor')

# Add data coverage flags
for col in ['gsc_impressions_total', 'ga4_sessions_total', 'word_count', 'search_volume', 'gsc_avg_position']:
    if col in baseline_scored.columns:
        vals = baseline_scored.set_index('content_hash_id')[col]
        queue[f'has_{col}'] = queue['content_hash_id'].map(vals).notna() & (queue['content_hash_id'].map(vals).fillna(0) > 0)

# Add supporting feature values
for col in ['gsc_impressions_total', 'gsc_clicks_total', 'gsc_avg_position', 'ga4_sessions_total',
            'word_count', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition_level',
            'content_type', 'main_intent']:
    if col in baseline_scored.columns:
        vals = baseline_scored.set_index('content_hash_id')[col]
        queue[col] = queue['content_hash_id'].map(vals)

# Add ranks
queue['model_rank'] = queue['best_model_probability'].rank(method='first', ascending=False).astype(int)
queue['baseline_rank'] = queue['baseline_score'].rank(method='first', ascending=False).astype(int)

# Add review category based on model probability
def review_category(prob):
    if prob >= 0.8: return 'high_priority_review'
    if prob >= 0.5: return 'medium_priority_review'
    return 'low_priority_monitor'
queue['review_category'] = queue['best_model_probability'].apply(review_category)

# Add human review note
queue['review_note'] = 'REQUIRES HUMAN REVIEW - model scores are decision support, not automatic instructions'

# Reorder columns
output_cols = ['model_rank', 'content_hash_id', 'client_hash_id', 'best_model_name',
               'best_model_probability', 'baseline_score', 'baseline_rank',
               'review_category', 'suggested_action', 'reason_codes',
               'is_declining_june', 'gsc_impressions_total', 'gsc_clicks_total',
               'gsc_avg_position', 'ga4_sessions_total', 'word_count',
               'content_age_days', 'days_since_last_update', 'search_volume',
               'competition_level', 'content_type', 'main_intent',
               'has_gsc_impressions_total', 'has_ga4_sessions_total',
               'has_word_count', 'has_search_volume', 'has_gsc_avg_position',
               'review_note']
output_cols = [c for c in output_cols if c in queue.columns]
queue = queue[output_cols].sort_values('model_rank')

# Save export
export_path = 'work/outputs/operational_review_queue.csv'
queue.to_csv(export_path, index=False)
print(f'Saved operational queue: {len(queue):,} rows to {export_path}')
print(f'Columns ({len(queue.columns)}): {list(queue.columns)}')
print()
print('=== Top 5 by model probability ===')
print(queue.head(5).to_string(index=False))
print()
print('=== Review category distribution ===')
print(queue['review_category'].value_counts().to_string())
print()
print('=== Suggested action distribution ===')
print(queue['suggested_action'].value_counts().to_string())
print()
print('=== Data coverage ===')
for col in ['has_gsc_impressions_total', 'has_ga4_sessions_total', 'has_word_count', 'has_search_volume', 'has_gsc_avg_position']:
    if col in queue.columns:
        print(f'  {col}: {queue[col].sum():,} / {len(queue):,} ({100*queue[col].mean():.1f}%)')

=== Building operational export queue ===



Saved operational queue: 63,500 rows to work/outputs/operational_review_queue.csv
Columns (28): ['model_rank', 'content_hash_id', 'client_hash_id', 'best_model_name', 'best_model_probability', 'baseline_score', 'baseline_rank', 'review_category', 'suggested_action', 'reason_codes', 'is_declining_june', 'gsc_impressions_total', 'gsc_clicks_total', 'gsc_avg_position', 'ga4_sessions_total', 'word_count', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition_level', 'content_type', 'main_intent', 'has_gsc_impressions_total', 'has_ga4_sessions_total', 'has_word_count', 'has_search_volume', 'has_gsc_avg_position', 'review_note']

=== Top 5 by model probability ===
 model_rank          content_hash_id          client_hash_id best_model_name  best_model_probability  baseline_score  baseline_rank      review_category suggested_action           reason_codes  is_declining_june  gsc_impressions_total  gsc_clicks_total  gsc_avg_position  ga4_sessions_total  word_count  content

## H. Reporting and limitations

### Validated results

- **Selected model (decision tree):** ROC AUC 0.8679, average precision 0.7908, P@20 0.90, P@50 0.90, P@100 0.86.
- **Random forest:** ROC AUC 0.8465, average precision 0.7846, P@20 1.00, P@50 0.98, P@100 0.96.
- **Heuristic baseline:** ROC AUC 0.7711, average precision 0.6640, P@20 0.75, P@50 0.84, P@100 0.77.
- **All models beat the heuristic baseline and dummy majority** on ROC AUC and average precision.
- **Feature ablation:** removing gsc_impressions_total does not change AUC (0.8679 -> 0.8679).
  Without all GSC features, AUC drops to 0.5485. GSC features are essential.
- **Client generalization:** consistent across 10 test clients (AUC 0.676-0.873). No single client dominates.
- **Metric verification:** all recomputed metrics match saved values (PASS).

### Baseline heuristic's distinct purpose

- The heuristic (baseline_score) is a **review-priority** score, not a decline predictor.
- It flags pages with high visibility, low CTR, staleness, or thin content for human review.
- Its Precision@K is below base rate for decline prediction (P@50: 0.84 vs base rate 0.48).
- This is expected: the heuristic prioritizes review candidates, not decline-prone pages.
- The model (decision tree) beats it at decline prediction (AUC 0.8679 vs 0.7711).

### Limitations

1. **gsc_impressions_total dominates feature importance (0.897).** Investigated in Phase 5: SAFE
   (legitimate signal, not leakage). But the model may over-rely on it. Worth monitoring.
2. **No time-aware backtest.** The client-grouped split is the only validation. A time-aware
   split would be more honest for deployment but is not fully feasible with the available data.
3. **Query-table features cover only May.** The `*_prev30` columns from query_90d cover only the
   30 days before the query window's end (May 2026). They don't cover the full feature window.
4. **No calibration tested.** Probability scores should be used for ranking only, not for
   threshold-based decisions without validation.
5. **Label balance by client.** 3 of 13 test clients have < 50 rows. Their metrics are unreliable.
6. **Dummy majority P@K.** Technically valid (predicts all positive) but not a meaningful
   ranking baseline. Genuine random ranking reference provided (P@50 mean=0.4806, std=0.006).
7. **No causal evaluation.** The model estimates risk, not treatment effect. No evidence that
   refreshing a page will prevent decline.
8. **Static metadata features** (content_type, search_volume, etc.) may not change over time,
   limiting the model's ability to detect temporal patterns.

### What the model does NOT claim

- The model does not predict Google's algorithm.
- The model does not guarantee that refreshing a page will improve performance.
- The model does not establish causation between any feature and decline.
- The model's probability scores are not calibrated.
- The model has not been validated on a time-aware split.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.

**Phase 6 summary:**
- Action playbook complete: 8 sections (A-H) + human review workflow + no-go list + monitoring triggers
- Operational queue exported: `work/outputs/operational_review_queue.csv`
- Reason codes consistent with `baseline_reason_codes.json`
- All metrics match verified saved artifacts
- No sample-pipeline results reported as warehouse results
- No commits, pushes, or publications made